# Workshop #1 - Data Profiling and Cross-Source Matching (Team 1: Earthquakes)
**Course:** Programación para Análisis de Datos  
**Author:** Team 1 - Seismic Activity Monitoring  

This notebook profiles 4 seismic data sources (2 historical, 2 incremental) and performs cross-source matching to identify candidate pairs representing the same physical earthquake event.

## 1. Environment Setup & Data Loading

In [4]:
import json
import xml.etree.ElementTree as ET
from pathlib import Path
import math
import pandas as pd
import numpy as np

SAMPLES_DIR = Path("../workshop_1/samples")

# 1. Load USGS Historical (GeoJSON)
with open(SAMPLES_DIR / "usgs_historical_raw.json", "r", encoding="utf-8") as f:
    usgs_hist_raw = json.load(f)

usgs_hist_list = []
for feat in usgs_hist_raw.get("features", []):
    props = feat.get("properties", {})
    coords = feat.get("geometry", {}).get("coordinates", [None, None, None])
    usgs_hist_list.append({
        "source": "USGS_Historical",
        "event_id": feat.get("id"),
        "timestamp_raw": props.get("time"),
        "timestamp_utc": pd.to_datetime(props.get("time"), unit="ms", utc=True) if props.get("time") else None,
        "latitude": coords[1],
        "longitude": coords[0],
        "depth_km": coords[2],
        "magnitude": props.get("mag"),
        "magnitude_type": props.get("magType"),
        "place": props.get("place")
    })
df_usgs_hist = pd.DataFrame(usgs_hist_list)

# 2. Load EMSC Historical (JSON/GeoJSON)
with open(SAMPLES_DIR / "emsc_historical_raw.json", "r", encoding="utf-8") as f:
    emsc_hist_raw = json.load(f)

emsc_hist_list = []
features_emsc = emsc_hist_raw.get("features", []) if isinstance(emsc_hist_raw, dict) else emsc_hist_raw
for feat in features_emsc:
    props = feat.get("properties", feat)
    geom = feat.get("geometry", {})
    coords = geom.get("coordinates", [props.get("lon"), props.get("lat"), props.get("depth")])
    emsc_hist_list.append({
        "source": "EMSC_Historical",
        "event_id": props.get("source_id") or feat.get("id"),
        "timestamp_raw": props.get("time"),
        "timestamp_utc": pd.to_datetime(props.get("time"), utc=True) if props.get("time") else None,
        "latitude": coords[1] if len(coords) > 1 else props.get("lat"),
        "longitude": coords[0] if len(coords) > 0 else props.get("lon"),
        "depth_km": coords[2] if len(coords) > 2 else props.get("depth"),
        "magnitude": props.get("mag"),
        "magnitude_type": props.get("magtype"),
        "place": props.get("flynn_region") or props.get("place")
    })
df_emsc_hist = pd.DataFrame(emsc_hist_list)

# 3. Load USGS Incremental (GeoJSON)
with open(SAMPLES_DIR / "usgs_incremental_raw.json", "r", encoding="utf-8") as f:
    usgs_inc_raw = json.load(f)

usgs_inc_list = []
for feat in usgs_inc_raw.get("features", []):
    props = feat.get("properties", {})
    coords = feat.get("geometry", {}).get("coordinates", [None, None, None])
    usgs_inc_list.append({
        "source": "USGS_Incremental",
        "event_id": feat.get("id"),
        "timestamp_raw": props.get("time"),
        "timestamp_utc": pd.to_datetime(props.get("time"), unit="ms", utc=True) if props.get("time") else None,
        "latitude": coords[1],
        "longitude": coords[0],
        "depth_km": coords[2],
        "magnitude": props.get("mag"),
        "magnitude_type": props.get("magType"),
        "place": props.get("place")
    })
df_usgs_inc = pd.DataFrame(usgs_inc_list)

# 4. Load GDACS Incremental (XML Feed)
tree = ET.parse(SAMPLES_DIR / "gdacs_incremental_raw.xml")
root = tree.getroot()
ns = {
    "geo": "http://www.w3.org/2003/01/geo/wgs84_pos#",
    "gdacs": "http://www.gdacs.org"
}
gdacs_list = []
for item in root.findall(".//item"):
    event_id = item.findtext("gdacs:eventid", namespaces=ns) or item.findtext("guid")
    pub_date = item.findtext("pubDate")
    lat = item.findtext("geo:lat", namespaces=ns)
    lon = item.findtext("geo:long", namespaces=ns)
    mag = item.findtext("gdacs:magnitude", namespaces=ns)
    gdacs_list.append({
        "source": "GDACS_Incremental",
        "event_id": event_id,
        "timestamp_raw": pub_date,
        "timestamp_utc": pd.to_datetime(pub_date, utc=True) if pub_date else None,
        "latitude": float(lat) if lat else None,
        "longitude": float(lon) if lon else None,
        "depth_km": None,  # Often absent in main XML RSS feed
        "magnitude": float(mag) if mag else None,
        "magnitude_type": "Mw",
        "place": item.findtext("title")
    })
df_gdacs_inc = pd.DataFrame(gdacs_list)

print(f"Loaded USGS Historical: {len(df_usgs_hist)} records")
print(f"Loaded EMSC Historical: {len(df_emsc_hist)} records")
print(f"Loaded USGS Incremental: {len(df_usgs_inc)} records")
print(f"Loaded GDACS Incremental: {len(df_gdacs_inc)} records")

Loaded USGS Historical: 26 records
Loaded EMSC Historical: 42 records
Loaded USGS Incremental: 189 records
Loaded GDACS Incremental: 22 records


## 2. Data Profiling & Quality Metrics

In [5]:
dfs = {
    "USGS_Historical": df_usgs_hist,
    "EMSC_Historical": df_emsc_hist,
    "USGS_Incremental": df_usgs_inc,
    "GDACS_Incremental": df_gdacs_inc
}

profile_results = []
for name, df in dfs.items():
    profile_results.append({
        "Source": name,
        "Total_Records": len(df),
        "Null_Magnitudes_%": (df["magnitude"].isna().mean() * 100),
        "Null_Depths_%": (df["depth_km"].isna().mean() * 100),
        "Min_Lat": df["latitude"].min(),
        "Max_Lat": df["latitude"].max(),
        "Min_Lon": df["longitude"].min(),
        "Max_Lon": df["longitude"].max(),
        "Unique_IDs": df["event_id"].nunique()
    })

df_profile = pd.DataFrame(profile_results)
display(df_profile)

,Source,Total_Records,Null_Magnitudes_%,Null_Depths_%,Min_Lat,Max_Lat,Min_Lon,Max_Lon,Unique_IDs
0,USGS_Historical,26,0.0,0.0,-56.1770,60.5558,-140.1891,168.6565,26
1,EMSC_Historical,42,0.0,0.0,-57.3317,60.5486,-140.0887,168.7216,42
2,USGS_Incremental,189,0.0,0.0,-27.6901,67.2690,-176.9956,179.1054,189
3,GDACS_Incremental,22,100.0,100.0,NaN,NaN,NaN,NaN,22


## 3. Documentation of 5 Concrete Data Quality Issues

1. **Heterogeneous Timestamp Representations:** USGS uses Epoch Unix milliseconds (`1772496000000`), EMSC uses ISO-8601 strings, and GDACS RSS uses RFC-822 formatted strings (`Mon, 02 Jan 2026 ...`).
2. **Missing Depth Measurements:** GDACS RSS feeds do not include hypocentral depth (`depth_km` = 100% null), whereas USGS/EMSC provide depth in kilometers.
3. **Inconsistent Magnitude Scales:** Sources mix different magnitude scales (`mb`, `mw`, `ml`, `md`) without explicit standardization.
4. **Informal Location Descriptions:** Descriptions vary from relative text (`"15 km ENE of Guayaquil, Ecuador"`) to regional names (`"NEAR COAST OF ECUADOR"`).
5. **Zero / Shallow Depth Distortions:** Micro-earthquakes or uncalibrated events often carry `depth_km = 0.0` or missing values requiring explicit quarantine handling.

## 4. Cross-Source Event Matching (Challenge for Team 1)

We calculate the geographical distance using the **Haversine formula** and match events between USGS and EMSC that occur within **100 km** and **$\pm 15$ minutes** of each other.

In [6]:
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371.0  # Earth radius in kilometers
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2)**2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2)**2
    return 2 * R * math.atan2(math.sqrt(a), math.sqrt(1 - a))

matches = []
for idx1, r1 in df_usgs_hist.iterrows():
    if pd.isna(r1["timestamp_utc"]) or pd.isna(r1["latitude"]):
        continue
    for idx2, r2 in df_emsc_hist.iterrows():
        if pd.isna(r2["timestamp_utc"]) or pd.isna(r2["latitude"]):
            continue
        
        # Calculate time difference in seconds
        time_diff = abs((r1["timestamp_utc"] - r2["timestamp_utc"]).total_seconds())
        
        # Window: 15 minutes (900 seconds)
        if time_diff <= 900:
            dist_km = haversine_distance(r1["latitude"], r1["longitude"], r2["latitude"], r2["longitude"])
            
            # Window: 100 km epicenter distance
            if dist_km <= 100:
                matches.append({
                    "USGS_ID": r1["event_id"],
                    "EMSC_ID": r2["event_id"],
                    "USGS_Time": r1["timestamp_utc"],
                    "EMSC_Time": r2["timestamp_utc"],
                    "Time_Diff_sec": round(time_diff, 1),
                    "Dist_km": round(dist_km, 2),
                    "USGS_Mag": r1["magnitude"],
                    "EMSC_Mag": r2["magnitude"],
                    "Mag_Diff": round(abs((r1["magnitude"] or 0) - (r2["magnitude"] or 0)), 2)
                })

df_matches = pd.DataFrame(matches).drop_duplicates(subset=["USGS_ID", "EMSC_ID"])
print(f"Found {len(df_matches)} candidate matching pairs across catalogs.")
display(df_matches.head(10))

Found 37 candidate matching pairs across catalogs.


,USGS_ID,EMSC_ID,USGS_Time,EMSC_Time,Time_Diff_sec,Dist_km,USGS_Mag,EMSC_Mag,Mag_Diff
0,us7000rre2,1923917,2026-01-01 23:27:18.120+00:00,2026-01-01 23:27:18.120+00:00,0.0,0.00,4.8,4.8,0.0
1,us7000rlzy,1923820,2026-01-01 22:49:46.878+00:00,2026-01-01 22:49:46.350+00:00,0.5,7.71,4.7,4.7,0.0
2,us7000rre0,1939364,2026-01-01 22:13:04.383+00:00,2026-01-01 22:13:04.383+00:00,0.0,0.00,4.5,4.5,0.0
3,us7000rlzn,1923794,2026-01-01 21:35:02.491+00:00,2026-01-01 21:35:01.650+00:00,0.8,6.99,5.2,5.2,0.0
4,us7000rlz9,1923771,2026-01-01 20:09:26.540+00:00,2026-01-01 20:09:27.150+00:00,0.6,4.92,4.5,4.5,0.0
5,us7000rlz4,1923760,2026-01-01 19:26:55.657+00:00,2026-01-01 19:26:56.490+00:00,0.8,13.98,4.7,4.7,0.0
6,us7000rrdv,1923781,2026-01-01 16:21:28.336+00:00,2026-01-01 16:21:29.400+00:00,1.1,68.12,4.5,4.5,0.0
7,us7000rrds,1923669,2026-01-01 15:30:25.540+00:00,2026-01-01 15:30:25.540+00:00,0.0,0.00,4.6,4.6,0.0
8,us7000rlzs,1923641,2026-01-01 14:16:38.715+00:00,2026-01-01 14:02:51.240+00:00,827.5,46.76,4.7,4.6,0.1
9,us7000rlxk,1923641,2026-01-01 14:02:50.757+00:00,2026-01-01 14:02:51.240+00:00,0.5,8.09,4.7,4.6,0.1
